# Topic Modeling
Unsupervised topic discovery on YouTube crisis comments using BERTopic.

## Methodology

### Motivation
After scoring comments by sentiment, we apply topic modeling to uncover
*what* people are talking about — blame attribution, calls for justice,
grief, misinformation, political critique — giving qualitative structure
to the quantitative sentiment scores.

### Approach
We use **BERTopic** — an embedding-based topic model that:
1. Encodes each comment into a dense vector (sentence-transformers)
2. Reduces dimensionality (UMAP) for clustering stability
3. Clusters semantically similar comments (HDBSCAN)
4. Extracts descriptive terms per cluster (c-TF-IDF)

### Pipeline

| Phase | Step | Description |
|---|---|---|
| **1. Preprocessing** | Text cleaning | Lowercase, strip punctuation/URLs/digits, remove English + Filipino stopwords |
| **2. Embedding** | Sentence vectors | `paraphrase-multilingual-MiniLM-L12-v2` (12 languages, 384-d) |
| **3. Reduction** | UMAP | 384-d → 5-d for density-based clustering |
| **4. Clustering** | HDBSCAN | Soft clustering; non-assigned → noise cluster (-1) |
| **5. Representation** | c-TF-IDF | Top unigrams/bigrams per cluster as topic descriptors |
| **6. Labeling** | Manual annotation | Human-readable labels from top terms + sample comments |

### Cross-Analysis
Topic assignments are joined with sentiment labels, shouting flags,
and video IDs to answer:
- *Which topics dominate each news source?*
- *Do certain topics carry more negative sentiment?*
- *Are shouted comments concentrated in specific topics?*

### Input / Output
- **Input:** `scored_{BATCH_NAME}.csv` from sentiment analysis
- **Output:** Topic-labeled DataFrame + per-video topic profile + topic × sentiment cross-tab

In [ ]:
import pandas as pd
import numpy as np
import ast
import os
import re
import string
import nltk

from nltk.corpus import stopwords
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

import spacy

import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.dpi"] = 150

## Configuration
Batch selector. Change `BATCH_NAME` to switch between
`youtube_comments_batch_{1,2,3}.csv`.

In [ ]:
# --- CONFIG ---
BASE_DIR = "../output"
BATCH_NAME = "scored_youtube_comments_batch_1.csv"
BATCH_PATH = os.path.join(BASE_DIR, BATCH_NAME)

## Load Batch
Read the scored CSV. Each row is a cleaned comment with
`sentiment_label`, `sentiment_score`, `is_shouting`, and
`sentences`. These features feed into cross-analysis
after topic modeling.

In [ ]:
batch_df = pd.read_csv(BATCH_PATH)
print(f"Loaded {len(batch_df)} comments from {BATCH_NAME}")

# Parse the sentences column from stringified list back to Python list
batch_df['sentences'] = batch_df['sentences'].apply(ast.literal_eval)

batch_df[["text", "sentiment_label", "sentiment_score", "is_shouting"]].head()


## Stopwords

Three sources merged into one set:

| Source | Provides | Origin |
|---|---|---|
| **nltk** | ~179 English stopwords | `nltk.corpus.stopwords` |
| **calamanCy** | ~147 Tagalog stopwords | spaCy defaults from `tl_calamancy_md` |
| **Custom** | ~30 noise words | `po`, `opo`, `kasi`, `lang`, `yung`, `nung`, etc. |

Custom additions cover common comment filler that isn't in
standard stopword lists.


In [ ]:
nltk.download("stopwords", quiet=True)

en_stop = set(stopwords.words("english"))
nlp_tl = spacy.load("tl_calamancy_md")
tl_stop = nlp_tl.Defaults.stop_words

custom_stop = {
    "po", "opo", "kasi", "eh", "ah", "oh", "talaga", "naman",
    "lang", "pa", "na", "din", "rin", "raw", "daw", "yung",
    "nung", "sya", "ung", "ng", "sa", "ako", "ikaw", "tayo",
    "niya", "natin", "sila", "amin", "atin",
}

STOPWORDS = en_stop | tl_stop | custom_stop
print(f"Stopwords: {len(STOPWORDS)} total")
print(f"  English:  {len(en_stop)}")
print(f"  Tagalog:  {len(tl_stop)}")
print(f"  Custom:   {len(custom_stop)}")
print(f"Sample TL: {sorted(tl_stop)[:10]}")


## Text Cleaning

Apply a lightweight cleaning pipeline for topic modeling:

- Lowercase
- Remove URLs
- Strip punctuation
- Remove digits
- Filter stopwords + tokens shorter than 3 characters

Cleaned text is stored in a new `text_clean` column.
The original `text` column is preserved.


In [ ]:
def clean_for_topics(text: str) -> str:
    text = text.lower()
    text = re.sub(r"https?://\S+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", "", text)
    tokens = [w for w in text.split() if w not in STOPWORDS and len(w) > 2]
    return " ".join(tokens)

batch_df["text_clean"] = batch_df["text"].apply(clean_for_topics)
corpus = batch_df["text_clean"].tolist()

print(f"Corpus: {len(corpus)} documents")
print(f"Avg tokens/doc before: {np.mean([len(t.split()) for t in batch_df['text']]):.0f}")
print(f"Avg tokens/doc after:  {np.mean([len(t.split()) for t in corpus]):.0f}")
